Reference: https://realpython.com/python-modules-packages/


Setting PATH to access python modules

In [1]:
import os, sys

# Go up THREE levels (parent of parent)
project_root = os.path.dirname(os.path.dirname(os.path.dirname(os.getcwd())))
# Go up N levels (general case)
# def go_up_n_dirs(current_path, n):
#     for _ in range(n):
#         current_path = os.path.dirname(current_path)
#     return current_path

# Append the new path to sys.path
if project_root not in sys.path:
    sys.path.append(project_root)
    print("Project root added to sys.path")
else:
    print("Project root already in sys.path")

Project root added to sys.path


In [2]:
# Checking
# sys.path

Import takes a while because of compilation from high-level $\longrightarrow$ bytecode files.

The bytecode files `.pyc` and `nbc` (JIT) are stored in `__pycache__` that is within `packages` directory. Now, any time those functions are called, is faster because skips compilation step and uses cached bytecode directly.

In [3]:
# Import takes a while since compile functions & compile JIT
from packages import oxygen_midpoint as om

# Parameters
dx = 2
box_length = 5
reciprocal_half_box = 1.0 / (0.5 * box_length) 

# Testing (success)
om.minimum_image_jit(3, box_length, reciprocal_half_box)

-2.0

---

Testing on real data

In [4]:
import numpy as np
import pandas as pd
from numba import njit

In [5]:
"""
Reads the npt-HK4.gro and returns the following:

data(pd.Dataframe): A dataframe that contains Residue ID(res_id) and Name(res_name), Atom Name (atom_name) and ID(atom_id), the atom coordinates(x, y, z) and velocity components(Vx, Vy, Vz)
title(str): The title of the .gro file
num_atoms(str): The number of atoms
box_dimensions(list): The simulation box dimensions in a list of [x,y,z] 

"""
# setting column specifications
colspecs = [
    (0, 5),
    (5, 8),
    (8, 15),
    (15, 20),
    (20, 28),
    (28, 36),
    (36, 44),
    (44, 52),
    (52, 60),
    (60, 68),
]

# setting names of columns
names = ["res_id", "res_name", "atom_name", "atom_id", "x", "y", "z", "Vx", "Vy", "Vz"]

# reading data
data = pd.read_fwf(
    "./../../../data/npt-HK4.gro",
    colspecs=colspecs,
    names=names,
    skiprows=2,
    skipfooter=1,
)

# Problem: after atom id 99999, it reverts back to 0. The following solves this:
mask = data.index >= 99_999
data.loc[mask, 'atom_id'] += 10_000
# data['atom_id'][mask] += 10_000 # Deprecated soon

# reading Title, number of atoms and box_dimensions
with open("../../../data/npt-HK4.gro", "rb") as f:
    title = f.readline().decode().strip()  # First line of .gro file
    num_atoms = f.readline().decode().strip()  # Second line of .gro file

    # Last line of file
    f.seek(-2, 2)
    while f.read(1) != b"\n":
        f.seek(-2, 1)
    box_dimensions = f.readline().decode().strip().split()
    box_dimensions = list(map(float, box_dimensions))


print(data.tail())
print(f"Title: {title}")
print(f"Number of atoms: {num_atoms}")
print(f"Box dimensions: {box_dimensions}")

        res_id res_name atom_name  atom_id      x      y      z      Vx  \
126079    1501      HK4       H23    36080  0.953  5.984  1.494 -1.2440   
126080    1501      HK4       C44    36081  0.736  6.009  1.525  0.1513   
126081    1501      HK4       H24    36082  0.707  6.041  1.425  0.3100   
126082    1501      HK4       C45    36083  0.632  5.988  1.615 -0.0110   
126083    1501      HK4       H25    36084  0.542  6.051  1.607 -0.7669   

            Vy      Vz  
126079  0.1068 -0.7226  
126080  0.5200  0.5632  
126081  2.4172  1.1123  
126082 -0.1637  0.2235  
126083 -0.9514  2.2265  
Title: mol only system in water
Number of atoms: 126084
Box dimensions: [11.24798, 11.24798, 11.24798]


In [6]:
# Dropping velocity components and setting index
data.drop(columns=["Vx", "Vy", "Vz"], inplace=True)  
data.set_index(['res_id', 'atom_name'], inplace=True) 

display(data)

res_name  atom_id      x      y      z
res_id atom_name                                       
1      H28            HK4        1  1.602  0.962  0.912
       C48            HK4        2  1.565  0.879  0.852
       C47            HK4        3  1.655  0.809  0.773
       H27            HK4        4  1.754  0.855  0.756
       C46            HK4        5  1.603  0.717  0.683
...                   ...      ...    ...    ...    ...
1501   H23            HK4    36080  0.953  5.984  1.494
       C44            HK4    36081  0.736  6.009  1.525
       H24            HK4    36082  0.707  6.041  1.425
       C45            HK4    36083  0.632  5.988  1.615
       H25            HK4    36084  0.542  6.051  1.607

[126084 rows x 5 columns]

In [7]:
# Extract box length from box_dimensions string 
box_length = box_dimensions[0]

# compute_midpoints_df_jit(data, box_length) # Eliminate overhead (NOT NECESSARY SINCE CACHED)
midpoints_df_jit = om.compute_midpoints_df_jit(data, box_length)
# %timeit _ = compute_midpoints_df_jit(data, box_length)

display(midpoints_df_jit)

,res_id,mid_x,mid_y,mid_z
0,1,1.55850,0.40900,11.16949
1,2,0.44150,0.65700,10.78750
2,3,0.76450,10.72000,0.89800
3,4,0.15751,0.97750,0.13500
4,5,5.54400,0.11051,5.77600
...,...,...,...,...
1496,1497,8.55200,6.78500,8.62900
1497,1498,8.60450,0.93600,7.08450
1498,1499,5.25950,0.52650,11.24649
1499,1500,5.92950,10.45100,3.18150
